# 🦀 Day 5: Rayon — Parallel Iterators

---

## What is Rayon?

Rayon is a **data parallelism** library for Rust. It makes parallelizing iterator chains almost trivial — just change `.iter()` to `.par_iter()` and you're done!

No manual thread spawning, no chunk splitting. Rayon handles work-stealing under the hood.

In [ ]:
:dep rayon = "1.10"

use rayon::prelude::*;

// Sequential vs Parallel — same API, different performance!
let nums: Vec<i32> = (1..=1_000_000).collect();

let seq_sum: i32 = nums.iter().sum();
let par_sum: i32 = nums.par_iter().sum();

println!("Sequential sum: {}", seq_sum);
println!("Parallel sum:   {}", par_sum);
assert_eq!(seq_sum, par_sum);

## Parallel Iterator Methods

| Method | Sequential | Parallel |
|--------|------------|----------|
| Iterate | `.iter()` | `.par_iter()` |
| Iterate mutably | `.iter_mut()` | `.par_iter_mut()` |
| Consume | `.into_iter()` | `.into_par_iter()` |

All the usual iterator adapters work: `map`, `filter`, `reduce`, `for_each`, etc.

In [ ]:
use rayon::prelude::*;

// par_iter, par_iter_mut, into_par_iter
let mut vec = vec![1, 2, 3, 4, 5, 6, 7, 8, 9, 10];

// par_iter_mut — modify in place
vec.par_iter_mut().for_each(|x| *x *= 2);
println!("Doubled: {:?}", vec);

// into_par_iter — consume and produce new collection
let squared: Vec<i32> = vec.into_par_iter().map(|x| x * x).collect();
println!("Squared: {:?}", squared);

## Parallel Map, Filter, Reduce

Chain adapters just like regular iterators. Rayon parallelizes the pipeline.

In [ ]:
use rayon::prelude::*;

let nums: Vec<i32> = (1..=100).collect();

// Parallel: filter evens, square, sum
let result: i32 = nums
    .par_iter()
    .filter(|&&x| x % 2 == 0)
    .map(|&x| x * x)
    .sum();

println!("Sum of squares of evens (1..100): {}", result);

// reduce — combine elements (must be associative)
let max = nums.par_iter().reduce(|| 0, |a, b| std::cmp::max(a, *b));
println!("Max: {}", max);

## par_sort() and par_chunks()

Rayon adds parallel sorting and chunking for collections.

In [ ]:
use rayon::prelude::*;

let mut data: Vec<i32> = (1..=20).rev().collect();
println!("Before sort: {:?}", data);

data.par_sort();
println!("After par_sort: {:?}", data);

// par_chunks — process chunks in parallel
let chunks: Vec<_> = data.par_chunks(5).map(|c| c.to_vec()).collect();
println!("Chunks of 5: {:?}", chunks);

## Parallelizing Word Frequency (from Month 1)

Recall the word counter from Month 1. We can parallelize it with Rayon!

In [ ]:
use rayon::prelude::*;
use std::collections::HashMap;

let text = "the quick brown fox jumps over the lazy dog the fox runs";
let words: Vec<&str> = text.split_whitespace().collect();

// Parallel: each chunk builds a local HashMap, then we merge
let freq: HashMap<&str, usize> = words
    .par_iter()
    .fold(
        || HashMap::new(),
        |mut map, word| {
            *map.entry(*word).or_insert(0) += 1;
            map
        },
    )
    .reduce(
        || HashMap::new(),
        |mut a, b| {
            for (k, v) in b {
                *a.entry(k).or_insert(0) += v;
            }
            a
        },
    );

println!("Word frequencies: {:?}", freq);

## Performance: Parallel vs Sequential

For CPU-bound work, parallel often wins. For tiny workloads, overhead can dominate.

In [ ]:
use rayon::prelude::*;
use std::time::Instant;

fn is_prime(n: u64) -> bool {
    if n < 2 { return false; }
    let mut i = 2u64;
    while i * i <= n {
        if n % i == 0 { return false; }
        i += 1;
    }
    true
}

let range: Vec<u64> = (2..=50_000).collect();

let start = Instant::now();
let seq_primes: Vec<u64> = range.iter().filter(|&&n| is_prime(n)).copied().collect();
let seq_time = start.elapsed();

let start = Instant::now();
let par_primes: Vec<u64> = range.par_iter().filter(|&&n| is_prime(n)).copied().collect();
let par_time = start.elapsed();

println!("Primes up to 50_000: {} found", seq_primes.len());
println!("Sequential: {:?}", seq_time);
println!("Parallel:   {:?}", par_time);
assert_eq!(seq_primes, par_primes);

## 🏋️ Exercises

In [ ]:
// Exercise: Use par_iter to find all primes in the range 2..=100_000.
// Collect them into a Vec<u64> and print the count.
// Hint: use the is_prime helper from above, or define your own.

use rayon::prelude::*;

fn is_prime(n: u64) -> bool {
    if n < 2 { return false; }
    let mut i = 2u64;
    while i * i <= n {
        if n % i == 0 { return false; }
        i += 1;
    }
    true
}

// YOUR CODE HERE


## 🎯 Key Takeaways

1. Rayon provides **data parallelism** — change `.iter()` to `.par_iter()` for easy parallelization
2. `par_iter()`, `par_iter_mut()`, `into_par_iter()` mirror standard iterator methods
3. `map`, `filter`, `reduce`, `for_each` all work in parallel
4. `par_sort()` and `par_chunks()` for parallel sorting and chunking
5. Use `fold` + `reduce` for parallel aggregation (e.g., word frequency)
6. Parallel wins for CPU-bound work; overhead matters for small inputs

---

**Tomorrow:** Atomic types (AtomicBool, AtomicUsize, Ordering) ⚛️